In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../../"))

# Install required packages
!pip install pythainlp wordcloud

In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
from pythainlp.corpus.common import thai_stopwords
from pythainlp.tokenize import word_tokenize

# Load data
df = pd.read_csv('../../../../data/raw/text/pizza-UTF8-traindataset-3.csv')
df = df.rename(columns={'class': 'class_label', 'message': 'text'})
print("Data loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Text preprocessing
thai_stopwords_list = list(thai_stopwords())

def clean_text(text):
    if pd.isna(text):
        return ""
    text = "".join(u for u in text if u not in ("?", ".", ";", ":", "!", '"', "ๆ", "ฯ", "'"))
    tokens = word_tokenize(text, engine="oskut", keep_whitespace=False)
    tokens = [word for word in tokens if word.lower not in thai_stopwords_list]
    tokens = [word for word in tokens if len(word) > 1]
    
    # Clean each token
    def process_token(word):
        word = word.strip().lower()
        word = word.translate(str.maketrans('', '', string.punctuation))
        return word if not word.isdigit() else ""
    
    tokens = [process_token(word) for word in tokens]
    tokens = [t for t in tokens if t]
    return ' '.join(tokens)

df['text'] = df['text'].apply(lambda x: clean_text(x))
print("Text preprocessing complete!")
df.head()

In [ ]:
# Train-test split
df_train, df_test = train_test_split(df, test_size=0.20, stratify=df['class_label'])

# Vectorize text
def identity_fun(text):
    return text.split()

vec = CountVectorizer(
    analyzer='word',
    tokenizer=identity_fun,
    ngram_range=(1, 3),
    max_features=300
)

X_train = vec.fit_transform(df_train['text'])
X_test = vec.transform(df_test['text'])

y_train = df_train['class_label']
y_test = df_test['class_label']

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
# Train model
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))